# Risk Parity - Long Only Equal Weight Strategy

This notebook demonstrates a long-only risk parity strategy using A-Stock ETFs with pysystemtrade.

**Instruments:**
- 510300.SH: Huatai-PB CSI 300 ETF (Equity)
- 518880.SH: Gold ETF (Commodity)
- 511260.SH: SSE Corporate Bond ETF (Fixed Income)

**Risk parity achieved through:**
1. Trading rule: constant 10.0 forecast (long bias)
2. Equal weights: pysystemtrade uses 1/n when no instrument_weights configured
3. Vol scaling: positionSize stage divides notional by instrument volatility

In [ ]:
# Setup: ensure target ETFs are in the universe

TARGET_ETFS = ["510300.SH", "518880.SH", "511260.SH"]

import pandas as pd
from pathlib import Path

# Check valid_symbols.csv
valid_symbols_path = Path("data/astock/csvconfig/valid_symbols.csv")
if valid_symbols_path.exists():
    valid_symbols = pd.read_csv(valid_symbols_path)
    existing = valid_symbols["Instrument"].tolist()
    missing = [e for e in TARGET_ETFS if e not in existing]
    if missing:
        print(f"Adding to valid_symbols.csv: {missing}")
        new_rows = pd.DataFrame({"Instrument": missing})
        valid_symbols = pd.concat([valid_symbols, new_rows], ignore_index=True)
        valid_symbols.to_csv(valid_symbols_path, index=False)
    else:
        print("All target ETFs already in valid_symbols.csv")
else:
    print("valid_symbols.csv not found, creating...")
    valid_symbols = pd.DataFrame({"Instrument": TARGET_ETFS})
    valid_symbols.to_csv(valid_symbols_path, index=False)

print(f"Target ETFs: {TARGET_ETFS}")

In [ ]:
# Fetch data for target ETFs

import subprocess

# Check if parquet data exists for target ETFs
parquet_dir = Path("data/astock/daily_prices_parquet")
existing_files = list(parquet_dir.glob("*.parquet")) if parquet_dir.exists() else []

print(f"Existing parquet files: {len(existing_files)}")

# Fetch data using the fetcher
print("\nFetching data for target ETFs...")
print("This may take a few minutes...")

result = subprocess.run(
    ["python", "-m", "sysinit.astock.fetcher", "--once", "--freq", "daily", "--universe", "all"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Data fetch completed!")
    if result.stdout:
        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
else:
    print(f"Error fetching data: {result.stderr}")
    print("\nYou may need to set XIXIMIAO_TOKEN in your .env file")

In [ ]:
# Verify data is available

from sysdata.sim.astock_sim_data import AStockSimData

data = AStockSimData()
available = data.get_instrument_list()

print(f"Available instruments: {len(available)}")

for etf in TARGET_ETFS:
    if etf in available:
        prices = data.get_raw_price(etf)
        print(f"  {etf}: {len(prices)} rows, {prices.index[0].date()} to {prices.index[-1].date()}")
    else:
        print(f"  {etf}: NOT AVAILABLE")

In [ ]:
# Create and run the risk parity system

from systems.basesystem import System
from systems.rawdata import RawData
from systems.trading_rules import TradingRule
from systems.provided.rules.long_only import long_only

# Create system with long_only trading rule
rule = TradingRule(long_only)
system = System([RawData(), rule], data=data)

print("System created successfully!")
print(f"Instruments in system: {system.get_instrument_list()}")

In [ ]:
# Display forecasts

print("=" * 60)
print("Risk Parity Strategy - Long Only Equal Weight")
print("=" * 60)

print("\nForecast (should be constant 10.0):")
for instr in TARGET_ETFS:
    if instr in system.get_instrument_list():
        forecast = system.rules.get_raw_forecast(instr, "long_only")
        print(f"  {instr}: {forecast.iloc[-1]:.4f}")
    else:
        print(f"  {instr}: not in system")

In [ ]:
# Display equal weights

print("\nEqual weights (1/n each):")
weights = system.portfolio.get_instrument_weights()
print(weights[TARGET_ETFS].tail())

In [ ]:
# Display vol-scaled positions

print("\nPosition sizes (vol-scaled):")
for instr in TARGET_ETFS:
    if instr in system.get_instrument_list():
        pos = system.positionSize.get_actual_position(instr)
        vol = system.positionSize.get_instrument_currency_vol(instr).iloc[-1]
        print(f"  {instr}: position={pos.iloc[-1]:.4f}, vol={vol:.6f}")